# Conferência dos valores publicados

Recalcula, a partir dos CSV brutos deste repositório, cada valor afirmado no
artigo, e confronta o resultado com o número publicado. A declaração de que os
dados sustentam o artigo não precisa ser aceita: ela é verificável aqui, linha a
linha.

A leitura dos arquivos e as estatísticas são refeitas neste notebook, sem
reaproveitar nada de `analise.py`. Duas implementações independentes chegando ao
mesmo número é uma verificação mais forte do que uma só executada duas vezes.

## Como executar

```bash
pip install numpy matplotlib
jupyter lab codigo/validacao.ipynb
```

Funciona igualmente no VS Code. O notebook localiza a raiz do repositório
sozinho, esteja ele aberto a partir de `codigo/` ou da raiz.

## O que este notebook não faz

Não valida a instrumentação. As amplitudes são **relativas**, tomadas com
analisador sem calibração vigente, sem detector de quase-pico e sem as larguras
de banda normativas. Nenhum valor aqui é medida de conformidade, e nenhum pode
ser confrontado com limite normativo expresso em dBµV/m. Ver as limitações no
`README.md`.

### Antes de rodar

O caderno recalcula tudo a partir dos CSV de `dados_brutos/`, então precisa
deles ao lado.

**Localmente** — abra a partir da raiz do repositório ou de `codigo/`; a
primeira célula encontra a pasta sozinha.

**No Google Colab** — a máquina começa vazia. A primeira célula cuida disso:
se houver uma URL de repositório, preencha `REPO` e ela clona; caso contrário
ela abre o seletor de arquivos e basta enviar um ZIP da pasta do repositório.
Não há nada a instalar: `numpy` e `matplotlib` já vêm no Colab.


In [ ]:
import io
import os
import sys

import numpy as np

# ------------------------------------------------------------------ origem
# O caderno precisa de dados_brutos/. Ele esta ao lado quando o repositorio
# foi baixado; no Colab, a maquina comeca vazia e os dados tem de chegar.
REPO = "https://github.com/DiyMV/emi-iluminacao-led-salas-cirurgicas.git"
ZIP_LOCAL = ""     # ou o caminho de um ZIP ja presente na sessao


def acha_raiz(inicio, niveis=3):
    """Procura a pasta que contem dados_brutos/, subindo e depois descendo."""
    p = os.path.abspath(inicio)
    for _ in range(niveis):
        if os.path.isdir(os.path.join(p, "dados_brutos")):
            return p
        pai = os.path.dirname(p)
        if pai == p:
            break
        p = pai
    for base, dirs, _ in os.walk(inicio):
        if "dados_brutos" in dirs:
            return base
        if base.count(os.sep) - inicio.count(os.sep) >= niveis:
            dirs[:] = []
    return None


def obtem_dados():
    """Traz o repositorio para a sessao do Colab. Devolve a raiz encontrada."""
    import subprocess
    import zipfile

    if REPO:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    elif ZIP_LOCAL:
        with zipfile.ZipFile(ZIP_LOCAL) as z:
            z.extractall("dados_enviados")
    else:
        # sem URL publicada: compacte a pasta do repositorio e envie o ZIP
        from google.colab import files
        for nome, dados in files.upload().items():
            with zipfile.ZipFile(io.BytesIO(dados)) as z:
                z.extractall("dados_enviados")
            print("extraido:", nome)
    return acha_raiz(os.getcwd())


NO_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

RAIZ = acha_raiz(os.getcwd())
if RAIZ is None and NO_COLAB:
    RAIZ = obtem_dados()

assert RAIZ is not None, (
    "dados_brutos/ nao encontrado. Fora do Colab, abra o caderno a partir da "
    "raiz do repositorio ou de codigo/. No Colab, preencha REPO ou envie o ZIP.")

# --------------------------------------------------------------- constantes
FCRIT = (30.0, 150.0)   # faixa de analise
FALTA = (225.0, 575.0)  # faixa que produz o falso negativo
CRIT = 3.0              # criterio de aceitacao, em dB
POS = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

print("python %s | numpy %s%s"
      % (sys.version.split()[0], np.__version__, "  | Colab" if NO_COLAB else ""))
print("raiz:", RAIZ)


## 1. Leitura

Formato de exportação do analisador: duas colunas por linha, frequência em hertz
e amplitude em dBm, sem cabeçalho. A frequência é convertida para MHz na leitura;
a amplitude fica como está.

A grandeza de interesse é a diferença entre as duas condições no mesmo ponto:

$$\Delta(f) = A_{\text{acesa}}(f) - A_{\text{apagada}}(f)$$

Os erros sistemáticos comuns às duas leituras — ganho de antena, perda de cabo,
desvio absoluto de calibração — cancelam-se nessa subtração. É o que torna a
comparação legítima apesar da ausência de calibração vigente.

In [ ]:
def ler(rel):
    """Devolve (f_MHz, amplitude_dBm) de uma varredura."""
    f, a = [], []
    caminho = os.path.join(RAIZ, rel)
    for linha in io.open(caminho, encoding="utf-8").read().splitlines():
        p = linha.replace(",", " ").split()
        if len(p) >= 2:
            try:
                f.append(float(p[0]) / 1e6)
                a.append(float(p[1]))
            except ValueError:
                pass
    return np.array(f), np.array(a)


def delta(rel_acesa, rel_apagada):
    """Elevacao no mesmo ponto. Recusa pares de grades diferentes."""
    f, a1 = ler(rel_acesa)
    g, a0 = ler(rel_apagada)
    assert np.allclose(f, g), "grades de frequencia diferentes: par nao comparavel"
    return f, a1 - a0


def media_faixa(f, y, lo, hi):
    """Media na faixa. Levanta excecao se a varredura nao a cobrir."""
    m = (f >= lo) & (f <= hi)
    assert m.sum() > 0, "a varredura nao cobre a faixa pedida"
    return float(y[m].mean())


f_, _ = ler("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_off.csv")
print("%d pontos, de %.1f a %.1f MHz, passo de %.1f kHz"
      % (len(f_), f_.min(), f_.max(), (f_[1] - f_[0]) * 1e3))

## 2. Campanha de verificação — três salas, sete pontos

Uma sala recebeu intervenção; as outras duas, medidas no mesmo dia com o mesmo
arranjo, funcionam como controle simultâneo. É essa simultaneidade — e não a
comparação temporal isolada — que sustenta a atribuição causal.

A reprovação de um único ponto reprova a sala.

In [ ]:
salas = {}
for s in ("01", "02", "03"):
    v = []
    for p in POS:
        base = "dados_brutos/campanha_2_verificacao/C2_sala%s_%s_luz_%%s.csv" % (s, p)
        f, d = delta(base % "on", base % "off")
        v.append(media_faixa(f, d, *FCRIT))
    salas[s] = np.array(v)

print("sala  media     min      max      dp    dispersao  veredito")
for s, v in salas.items():
    print("%s   %+7.2f  %+7.2f  %+7.2f  %5.2f   %6.2f     %s"
          % (s, v.mean(), v.min(), v.max(), v.std(ddof=1), np.ptp(v),
             "aprovado" if v.max() <= CRIT else "REPROVADO"))

print("\ndispersao espacial: emissao distribuida por todo o circuito produz")
print("pouca variacao entre pontos; contribuicao localizada, muita.")

## 3. Campanha de diagnóstico — a falha clinicamente ativa

Cinco varreduras sucessivas contra uma referência, antes da intervenção. O
desvio-padrão entre elas caracteriza a **repetibilidade de curto prazo** do
arranjo — e é essa, não o intervalo de calibração, a grandeza de estabilidade
relevante para uma diferença tomada no intervalo de segundos.

In [ ]:
REF = "dados_brutos/campanha_1_diagnostico/C1_sala03_luz_off_ref.csv"
ON = "dados_brutos/campanha_1_diagnostico/C1_sala03_luz_on_rep%d.csv"

c1 = []
for i in range(1, 6):
    f1, d1 = delta(ON % i, REF)
    c1.append(media_faixa(f1, d1, *FCRIT))
c1 = np.array(c1)

print("varreduras:", "  ".join("%+.2f" % x for x in c1))
print("media %+.2f dB   dp %.2f dB" % (c1.mean(), c1.std(ddof=1)))
print("efeito sobre repetibilidade: %.0f vezes" % (c1.mean() / c1.std(ddof=1)))

## 4. O efeito da faixa varrida

Mesma sala, mesmo ponto, mesmo dia: muda apenas a faixa medida. O resultado
inverte. Uma varredura na faixa alta aprovaria uma sala que a faixa crítica
reprova por margem de mais de quatro vezes.

**A escolha da faixa é parte do critério de aceitação, não um detalhe de
configuração.**

In [ ]:
faixa = {}
for s in ("01", "03"):
    alta = "dados_brutos/demonstracao_faixa/DF_sala%s_luz_%%s_225-575MHz.csv" % s
    crit = "dados_brutos/campanha_2_verificacao/C2_sala%s_P4_luz_%%s.csv" % s
    fa, da = delta(alta % "on", alta % "off")
    fc, dc = delta(crit % "on", crit % "off")
    faixa[s] = (media_faixa(fa, da, *FALTA), media_faixa(fc, dc, *FCRIT))

print("sala   225-575 MHz   30-150 MHz   subestimacao")
for s, (alta, crit) in faixa.items():
    print("%s      %+8.2f      %+8.2f      %8.2f" % (s, alta, crit, crit - alta))

## 5. A fronteira de 80 MHz

O ensaio de imunidade a campos de RF **radiados** (IEC 61000-4-3) da norma
colateral de compatibilidade eletromagnética para equipamento eletromédico
começa em 80 MHz. Abaixo dessa fronteira, a imunidade do equipamento é
qualificada pelo ensaio de **perturbações conduzidas induzidas por RF**
(IEC 61000-4-6), cujo caminho de acoplamento é o cabeamento e não o campo
incidente — é a qualificação do equipamento, não a natureza da emissão medida,
que é radiada.

Quanto do fenômeno medido cai de cada lado dessa fronteira?

In [ ]:
fc1, _ = ler(REF)
dm = np.vstack([delta(ON % i, REF)[1] for i in range(1, 6)]).mean(axis=0)

m = (fc1 >= FCRIT[0]) & (fc1 <= FCRIT[1])
baixo, alto = m & (fc1 < 80), m & (fc1 >= 80)

print("pontos espectrais na faixa de analise: %d" % m.sum())
print("  30-80  MHz: %3d (%.0f%%)  media %+.2f dB   imunidade conduzida (IEC 61000-4-6)"
      % (baixo.sum(), 100 * baixo.sum() / m.sum(), dm[baixo].mean()))
print("  80-150 MHz: %3d (%.0f%%)  media %+.2f dB   imunidade radiada (IEC 61000-4-3)"
      % (alto.sum(), 100 * alto.sum() / m.sum(), dm[alto].mean()))

ordem = np.argsort(dm[m])[::-1][:10]
print("\ndez maiores elevacoes:")
for k in ordem:
    print("  %8.2f MHz   %+6.2f dB %s"
          % (fc1[m][k], dm[m][k], "<- abaixo de 80 MHz" if fc1[m][k] < 80 else ""))

print("\nEsta observacao nao constata violacao de requisito: as amplitudes sao")
print("relativas e nao admitem confronto com nivel de imunidade em V/m.")

## 6. Conferência

Cada linha é uma afirmação publicada. `tol` é a tolerância de arredondamento do
próprio texto. Qualquer REPROVA é divergência entre o artigo e estes dados —
e, sendo esse o caso, o erro é do artigo, não seu.

In [ ]:
fp, dp = delta("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_on.csv",
               "dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_off.csv")
mp = (fp >= FCRIT[0]) & (fp <= FCRIT[1])

conferencia = [
    ("sala 01 - media",              salas["01"].mean(),         12.68, .01),
    ("sala 01 - minimo",             salas["01"].min(),          11.62, .01),
    ("sala 01 - maximo",             salas["01"].max(),          13.77, .01),
    ("sala 01 - desvio-padrao",      salas["01"].std(ddof=1),     0.70, .01),
    ("sala 01 - dispersao",          np.ptp(salas["01"]),         2.15, .01),
    ("sala 02 - media",              salas["02"].mean(),         12.40, .01),
    ("sala 02 - dispersao",          np.ptp(salas["02"]),         5.08, .01),
    ("sala 03 - media",              salas["03"].mean(),          1.04, .01),
    ("sala 03 - maximo",             salas["03"].max(),           1.73, .01),
    ("diagnostico - media",          c1.mean(),                  13.23, .01),
    ("diagnostico - repetibilidade", c1.std(ddof=1),              0.89, .01),
    ("faixa alta, sala 01",          faixa["01"][0],              2.94, .01),
    ("faixa critica, sala 01",       faixa["01"][1],             13.01, .01),
    ("subestimacao, sala 01",        faixa["01"][1] - faixa["01"][0], 10.07, .02),
    ("faixa alta, sala 03",          faixa["03"][0],              0.07, .01),
    ("faixa critica, sala 03",       faixa["03"][1],              0.47, .01),
    ("80 MHz - media 30-80",         dm[baixo].mean(),           13.65, .01),
    ("80 MHz - media 80-150",        dm[alto].mean(),            12.93, .01),
    ("80 MHz - pontos abaixo",       float(baixo.sum()),         64.00, .00),
    ("80 MHz - pontos na faixa",     float(m.sum()),            154.00, .00),
    ("pico da sala 01",              dp[mp].max(),               30.53, .01),
    ("frequencia do pico",           fp[mp][int(np.argmax(dp[mp]))], 81.85, .01),
]

# Nem toda linha acima e afirmacao do artigo. Estas quatro sao grandezas
# derivadas, conferidas aqui por completude: o artigo nao as publica, e
# procura-las nele seria busca infrutifera -- com a suspeita recaindo sobre
# o artigo, que nada afirmou.
DERIVADAS = {
    "sala 01 - desvio-padrao",   # o artigo publica o desvio da repetibilidade
    "faixa alta, sala 03",       # o efeito de faixa e tratado so na sala 01
    "faixa critica, sala 03",
    "frequencia do pico",        # publica o valor do pico, nao sua frequencia
}

falhas = 0
print("%-30s %10s %10s %8s  %-8s %s"
      % ("afirmacao", "calculado", "esperado", "erro", "fonte", "situacao"))
print("-" * 84)
for nome, calc, pub, tol in conferencia:
    erro = abs(calc - pub)
    ok = erro <= tol
    falhas += (not ok)
    print("%-30s %10.2f %10.2f %8.3f  %-8s %s"
          % (nome, calc, pub, erro,
             "derivada" if nome in DERIVADAS else "artigo",
             "ok" if ok else "REPROVA"))
print("-" * 84)
n_art = len(conferencia) - len(DERIVADAS)
print("%d afirmacoes conferidas, %d divergentes" % (len(conferencia), falhas))
print("   %d constam do artigo; %d sao derivadas, conferidas por completude"
      % (n_art, len(DERIVADAS)))

## 7. O que a faixa de análise decide

O painel **(a)** mostra os dois níveis absolutos, em dBm: a curva com a
iluminação acionada e a curva com ela apagada. A distância média entre as
duas, na faixa crítica, é o número que o artigo publica.

O painel **(b)** é essa mesma distância, calculada bin a bin. Repare que a
condição apagada não aparece nele — ela está *dentro* da subtração, e o eixo
passa a ser razão (dB), não nível (dBm). É a confusão mais comum na leitura
deste tipo de figura.

A área sombreada em (b) é o $\Delta$ **médio** de cada faixa, que é a grandeza
avaliada pelo critério. Pontos espectrais isolados chegam a mais que o dobro
dessa média — e é por isso que o critério não é aplicado a bins individuais.

In [ ]:
import matplotlib.pyplot as plt

alta = "dados_brutos/demonstracao_faixa/DF_sala01_luz_%s_225-575MHz.csv"
fa, a1 = ler(alta % "on")
_, a0 = ler(alta % "off")
da = a1 - a0
fx, ac = ler("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_on.csv")
_, ap = ler("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_off.csv")

m_crit, m_alta = faixa["01"][1], faixa["01"][0]
mc = (fx >= FCRIT[0]) & (fx <= FCRIT[1])
n_on, n_off = ac[mc].mean(), ap[mc].mean()
VM, AZ, CZ, TI = "#d03b3b", "#2a78d6", "#8a8f98", "#1a1d21"

fig, (axa, ax) = plt.subplots(2, 1, figsize=(7.6, 5.4), sharex=True, dpi=120,
                              gridspec_kw=dict(height_ratios=[1, 1.12],
                                               hspace=.12))
for eixo in (axa, ax):
    eixo.axvspan(*FCRIT, color=VM, alpha=.085, lw=0)
    eixo.axvspan(*FALTA, color=AZ, alpha=.075, lw=0)
    for lado in ("top", "right"):
        eixo.spines[lado].set_visible(False)

# (a) os dois niveis absolutos. Sem este painel o leitor pergunta onde esta
# a curva de luz apagada -- e a pergunta e boa: ela esta dentro da subtracao
for f_, acesa, apagada in ((fx, ac, ap), (fa, a1, a0)):
    axa.fill_between(f_, apagada, acesa, where=acesa >= apagada,
                     color=CZ, alpha=.18, lw=0)
    axa.plot(f_, acesa, color=TI, lw=.85)
    axa.plot(f_, apagada, color=CZ, lw=.7)
axa.plot(FCRIT, [n_on] * 2, color=TI, lw=2.4, solid_capstyle="butt")
axa.plot(FCRIT, [n_off] * 2, color=CZ, lw=2.4, solid_capstyle="butt")
axa.annotate("", xy=(152, n_on), xytext=(152, n_off),
             arrowprops=dict(arrowstyle="<|-|>", color=VM, lw=1.2))
axa.text(160, (n_on + n_off) / 2, "%+.2f dB" % m_crit, fontsize=10,
         color=VM, weight="bold", va="center")
ha = [plt.Line2D([], [], color=TI, lw=1.4),
      plt.Line2D([], [], color=CZ, lw=1.4),
      plt.Line2D([], [], color=CZ, lw=2.6, alpha=.55)]
axa.legend(ha, ["luz acesa", "luz apagada", "media em 30-150 MHz"],
           loc="upper right", frameon=False, fontsize=9,
           handlelength=1.4, labelspacing=.25, borderaxespad=.35)
util = fx >= 25   # abaixo de ~5 MHz e raia de DC do instrumento, nao sinal
axa.set_ylim(min(ap[util].min(), a0.min()) - 3,
             max(ac[util].max(), a1.max()) + 7)
axa.set_ylabel("Nivel (dBm)")

# (b) a mesma informacao como diferenca: o eixo aqui e razao, nao nivel
# num grafico de diferenca a condicao apagada e identicamente zero:
# subtraida de si mesma. Nomear a linha evita que o leitor procure
# por uma curva de referencia que nao pode existir aqui
ax.axhline(0, color=CZ, lw=1.0)
ax.fill_between(FCRIT, 0, m_crit, color=VM, alpha=.26, lw=0)
ax.fill_between(FALTA, 0, m_alta, color=AZ, alpha=.26, lw=0)
ax.plot(fx, ac - ap, color=VM, lw=.4, alpha=.22)
ax.plot(fa, da, color=AZ, lw=.4, alpha=.22)
ax.plot(FCRIT, [m_crit] * 2, color=VM, lw=3, solid_capstyle="butt")
ax.plot(FALTA, [m_alta] * 2, color=AZ, lw=3, solid_capstyle="butt")
ax.axhline(CRIT, color="#4a4f57", lw=1, ls=(0, (4, 2)))
ax.text(4, CRIT + 1.0, "criterio %.0f dB" % CRIT, ha="left",
        va="bottom", fontsize=9, color="#4a4f57",
        bbox=dict(boxstyle="round,pad=.2", fc="white", ec="none",
                  alpha=.8))
# uma faixa por entrada de legenda: nao ha largura para dois rotulos
NL = chr(10)   # quebra de linha dentro do rotulo da legenda
hb = [plt.Line2D([], [], color=VM, lw=1.4),
      plt.Line2D([], [], color=AZ, lw=1.4),
      plt.Line2D([], [], color=CZ, lw=1.4)]
rot = ["%g-%g MHz" % FCRIT + NL + "Δ %+.2f dB - reprova" % m_crit,
       "%g-%g MHz" % FALTA + NL + "Δ %+.2f dB - aprova" % m_alta,
       "luz apagada" + NL + "referencia = 0 dB"]
ax.legend(hb, rot, loc="upper right", frameon=False, fontsize=9,
          handlelength=1.4, labelspacing=.55, borderaxespad=.35)
ax.set_ylim(-9, 44)
ax.set_yticks([-5, 0, 5, 10, 15, 20, 25, 30])
ax.set_xlim(0, 580)
ax.set_xlabel("Frequencia (MHz)")
ax.set_ylabel("Δ  acesa - apagada (dB)")

for eixo, rot in ((axa, "(a)"), (ax, "(b)")):
    eixo.text(.012, .95, rot, transform=eixo.transAxes, fontsize=10,
              weight="bold", va="top")
fig.tight_layout()
plt.show()

## 8. As figuras do artigo, reconstruídas

As duas figuras de espectro do artigo são redesenhadas aqui a partir dos
mesmos CSV — nenhuma imagem é importada. Cada uma aparece nas duas versões
que o artigo mantém: a **isométrica**, publicada, e a **plana**, preservada
como alternativa. O par existe porque a isométrica separa condições que se
sobrepõem na planar, e a planar permite ler amplitude contra o eixo.

A paleta é a do artigo, verificada para deuteranopia e protanopia: a
distância mínima entre as cores, depois de simulada a visão dicromática,
supera 20 unidades CIE76, e as duas curvas que precisam ser distinguidas
diferem também em luminância — quem não separa matiz ainda separa claro de
escuro.


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# paleta do artigo -------------------------------------------------------
LARANJA = "#ff2d1a"     # condicao acesa, antes da mitigacao
AZUL = "#2a78d6"        # condicao posterior, e a faixa que aprova
AMARELO_Q = "#4a3600"   # condicao apagada, referencia
VERMELHO = "#d03b3b"    # faixa de analise, marcada no chao
VERDE_LIMA = "#c6f000"  # caixa do criterio
TINTA = "#1a1d21"
CINZA = "#8a8f98"


def suave(y, n=5):
    """Media movel curta; o dado bruto vira textura densa e esconde a
    conclusao.

    Modo 'valid' com preenchimento das pontas: o modo 'same' do convolve
    introduz artefato de borda, que aqui apareceria como queda vertical
    espuria no primeiro e no ultimo bin de cada varredura.
    """
    if n < 2:
        return y
    v = np.convolve(y, np.ones(n) / n, mode="valid")
    m = n // 2
    return np.concatenate([np.full(m, v[0]), v,
                           np.full(len(y) - len(v) - m, v[-1])])


def fita(x, y, z, base):
    """Poligono vertical sob a curva, para dar corpo a serie na isometrica."""
    v = [(x[0], y, base)]
    v += [(xi, y, zi) for xi, zi in zip(x, z)]
    v += [(x[-1], y, base)]
    return [v]


def chao(x0, x1, y0, y1, z):
    return [[(x0, y0, z), (x1, y0, z), (x1, y1, z), (x0, y1, z)]]


def moldura(ax, ymax, zlim, xticks=True, largura=3.0, zoom=1.28):
    """Projecao ortografica: sem perspectiva, alturas ficam comparaveis."""
    ax.set_proj_type("ortho")
    ax.patch.set_alpha(0)
    ax.view_init(elev=20.0, azim=-64.0)
    ax.set_box_aspect((largura, 1.0, 1.0), zoom=zoom)
    ax.set_xlim(-6, 378)
    ax.set_ylim(-.4, ymax)
    ax.set_zlim(*zlim)
    ax.set_xticks([0, 100, 200, 300, 350])
    if not xticks:
        ax.set_xticklabels([])
    ax.set_yticks([])
    ax.tick_params(labelsize=7, pad=-1.5, colors="#3b4048")
    for eixo in (ax.xaxis, ax.yaxis, ax.zaxis):
        eixo.pane.fill = False
        eixo.pane.set_edgecolor("white")
    ax.grid(color="#eceef1", linewidth=.35, alpha=.75)


def limpa(ax):
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(labelsize=8)


def posiciona(axa, axb):
    """Estica os dois eixos sobre a figura.

    O cubo isometrico ocupa uma fracao pequena da caixa do subplot: deixado
    no lugar padrao, sobra margem vazia dos dois lados e o desenho sai
    espremido. Sao as mesmas proporcoes usadas na composicao do artigo.
    """
    axa.set_position([-.20, .33, 1.30, .70])
    axb.set_position([-.20, .02, 1.30, .70])


def rotulos(fig, a, b):
    fig.text(.06, .965, a, fontsize=10, fontweight="bold", color=TINTA)
    fig.text(.06, .478, b, fontsize=10, fontweight="bold", color=TINTA)


print("auxiliares de figura prontos")


### Figura 1 — o que a faixa de análise decide

Mesma sala, mesmo ponto, mesmo dia; muda apenas a faixa medida, e o veredito inverte.


In [ ]:
# ---------------------------------------------------------------- dados
f1, ac1 = ler("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_on.csv")
_, ap1 = ler("dados_brutos/campanha_2_verificacao/C2_sala01_P4_luz_off.csv")
fA, acA = ler("dados_brutos/demonstracao_faixa/DF_sala01_luz_on_225-575MHz.csv")
_, apA = ler("dados_brutos/demonstracao_faixa/DF_sala01_luz_off_225-575MHz.csv")

# o vazamento de DC do analisador domina os primeiros bins: artefato de
# instrumento, nao emissao da sala
u = f1 >= 2.0
fx, acesa, apagada = f1[u], suave(ac1[u]), suave(ap1[u])
delta = suave((ac1 - ap1)[u])

mc = (f1 >= FCRIT[0]) & (f1 <= FCRIT[1])
mA = (fA >= FALTA[0]) & (fA <= FALTA[1])
m_crit = (ac1 - ap1)[mc].mean()
m_alta = (acA - apA)[mA].mean()
n_on, n_off = ac1[mc].mean(), ap1[mc].mean()

# ------------------------------------------------------- versao isometrica
fig = plt.figure(figsize=(9.5, 5.4), facecolor="white")

piso = apagada.min() - 3.0
axa = fig.add_subplot(2, 1, 1, projection="3d")
axa.add_collection3d(Poly3DCollection(fita(fx, 0, apagada, piso),
                                      alpha=.22, facecolor=AMARELO_Q, lw=0))
axa.add_collection3d(Poly3DCollection(fita(fx, 1, acesa, piso),
                                      alpha=.22, facecolor=LARANJA, lw=0))
axa.plot(fx, np.ones_like(fx), acesa, color=LARANJA, lw=1.0, label="luz acesa")
axa.plot(fx, np.zeros_like(fx), apagada, color=AMARELO_Q, lw=.9,
         label="luz apagada")
for (lo, hi), cor in (((FCRIT[0], FCRIT[1]), VERMELHO), ((225, 350), AZUL)):
    axa.add_collection3d(Poly3DCollection(chao(lo, hi, -.4, 1.4, piso),
                                          alpha=.16, facecolor=cor, lw=0))
moldura(axa, 1.4, (piso, acesa.max() + 2), xticks=False)
axa.legend(loc="upper right", bbox_to_anchor=(.90, 1.015),
           bbox_transform=fig.transFigure, frameon=False, fontsize=8,
           handlelength=1.4, labelspacing=.2)

pisob = min(delta.min(), -2.0) - 2.0
axb = fig.add_subplot(2, 1, 2, projection="3d")
axb.add_collection3d(Poly3DCollection(fita(fx, 0, delta, pisob),
                                      alpha=.22, facecolor=LARANJA, lw=0))
axb.plot(fx, np.zeros_like(fx), delta, color=LARANJA, lw=1.0)
for (lo, hi), cor in (((FCRIT[0], FCRIT[1]), VERMELHO), ((225, 350), AZUL)):
    axb.add_collection3d(Poly3DCollection(chao(lo, hi, -.4, .4, pisob),
                                          alpha=.16, facecolor=cor, lw=0))
axb.plot([], [], [], color=VERMELHO, lw=1.2, alpha=.6,
         label="%g–%g MHz · reprova" % FCRIT)
axb.plot([], [], [], color=AZUL, lw=1.2, alpha=.6, label="225–350 MHz · aprova")
moldura(axb, .4, (pisob, max(delta.max(), CRIT) + 11))
axb.legend(loc="lower left", bbox_to_anchor=(.035, .012),
           bbox_transform=fig.transFigure, frameon=False, fontsize=8,
           handlelength=1.4, labelspacing=.2)
fig.text(.72, .045, "Frequência (MHz)", fontsize=9, color=TINTA, ha="center")
posiciona(axa, axb)
rotulos(fig, "(a)  Nível (dBm)", "(b)  Elevação Δ (dB)")
print("Figura 1 — versão isométrica, como publicada no artigo")
plt.show()

# ----------------------------------------------------------- versao plana
fig, (pa, pb) = plt.subplots(2, 1, figsize=(9.2, 5.6), sharex=True, dpi=110,
                             gridspec_kw=dict(hspace=.14))
for eixo in (pa, pb):
    eixo.axvspan(*FCRIT, color=VERMELHO, alpha=.085, lw=0)
    eixo.axvspan(*FALTA, color=AZUL, alpha=.075, lw=0)
    limpa(eixo)

for f_, a_, p_ in ((f1[u], acesa, apagada), (fA, suave(acA), suave(apA))):
    pa.fill_between(f_, p_, a_, where=a_ >= p_, color=VERMELHO, alpha=.13, lw=0)
    pa.plot(f_, a_, color=VERMELHO, lw=.9)
    pa.plot(f_, p_, color=CINZA, lw=.8)
pa.plot(FCRIT, [n_on] * 2, color=VERMELHO, lw=2.0, solid_capstyle="butt")
pa.plot(FCRIT, [n_off] * 2, color=CINZA, lw=2.0, solid_capstyle="butt")
pa.annotate("", xy=(152, n_on), xytext=(152, n_off),
            arrowprops=dict(arrowstyle="<|-|>", color=VERMELHO, lw=1.0))
pa.text(160, (n_on + n_off) / 2, "%+.2f dB" % m_crit, va="center",
        fontsize=9, color=VERMELHO, fontweight="bold")
pa.set_ylabel("Nível (dBm)")
pa.set_title("(a)", loc="left", fontsize=10, fontweight="bold")

pb.axhline(0, color=CINZA, lw=.8)
pb.fill_between(FCRIT, 0, m_crit, color=VERMELHO, alpha=.26, lw=0)
pb.fill_between(FALTA, 0, m_alta, color=AZUL, alpha=.26, lw=0)
pb.plot(f1[u], delta, color=VERMELHO, lw=.9)
pb.plot(fA, suave(acA - apA), color=AZUL, lw=.9)
pb.axhline(CRIT, color="#4a4f57", lw=.8, ls=(0, (3.5, 2)))
pb.text(4, CRIT + 1.2, "critério %g dB" % CRIT, fontsize=8, color=TINTA)
pb.set_ylabel("Δ acesa − apagada (dB)")
pb.set_xlabel("Frequência (MHz)")
pb.set_title("(b)", loc="left", fontsize=10, fontweight="bold")
pb.legend([plt.Line2D([], [], color=VERMELHO, lw=1.2),
           plt.Line2D([], [], color=AZUL, lw=1.2)],
          ["%g–%g MHz · Δ %+.2f dB · reprova" % (FCRIT[0], FCRIT[1], m_crit),
           "%g–%g MHz · Δ %+.2f dB · aprova" % (FALTA[0], FALTA[1], m_alta)],
          loc="upper right", frameon=False, fontsize=8)
fig.tight_layout()
print("Figura 1 — versão plana, preservada no artigo como alternativa")
plt.show()

print("faixa de análise %+.2f dB   faixa alta %+.2f dB   subestimação %.2f dB"
      % (m_crit, m_alta, m_crit - m_alta))


### Figura 3 — a sala corrigida, contra ela mesma

A condição anterior, a posterior e a referência com as luminárias apagadas.


In [ ]:
# ---------------------------------------------------------------- dados
# A condicao posterior e a media das sete posicoes, nao um ponto isolado:
# e essa media que o artigo publica como +1,04 dB. Tomar so o ponto central
# daria outro numero, e a figura deixaria de corresponder ao texto.
def malha(sala, cond):
    v = [ler("dados_brutos/campanha_2_verificacao/C2_sala%s_%s_luz_%s.csv"
             % (sala, p, cond)) for p in POS]
    return v[0][0], np.mean([a for _, a in v], axis=0)


fr, v_ref = ler(REF)
v_antes = np.vstack([ler(ON % i)[1] for i in range(1, 6)]).mean(axis=0)
fd, v_dep = malha("03", "on")
_, v_dref = malha("03", "off")

d_antes, d_depois = v_antes - v_ref, v_dep - v_dref
ma = (fr >= FCRIT[0]) & (fr <= FCRIT[1])
md = (fd >= FCRIT[0]) & (fd <= FCRIT[1])
m_a, m_d = d_antes[ma].mean(), d_depois[md].mean()

# Criterio declarado no artigo: maximo local da ELEVACAO acima da media da
# faixa mais um desvio-padrao. Aplicar ao nivel absoluto daria outro
# conjunto -- o que se conta e o quanto a iluminacao acrescentou, nao o
# quanto o ambiente ja tinha.
lim = d_antes[ma].mean() + d_antes[ma].std()
idx = [k for k in range(1, len(fr) - 1)
       if ma[k] and d_antes[k] > lim
       and d_antes[k] > d_antes[k - 1] and d_antes[k] > d_antes[k + 1]]
print("máximos locais na faixa de análise: %d  (entre %.1f e %.1f MHz)"
      % (len(idx), fr[idx].min(), fr[idx].max()))

# o vazamento de corrente continua do analisador domina os primeiros bins:
# sem cortar, ele vira um pico vertical que nao e emissao da sala
cr, cd = fr >= 2.0, fd >= 2.0

# ------------------------------------------------------- versao isometrica
fig = plt.figure(figsize=(9.5, 5.4), facecolor="white")
piso = -102.0
axa = fig.add_subplot(2, 1, 1, projection="3d")
for y, f_, s_, cor, rot in ((0, fr[cr], v_ref[cr], AMARELO_Q, "apagada — referência"),
                            (1, fd[cd], v_dep[cd], AZUL, "acesa — depois"),
                            (2, fr[cr], v_antes[cr], LARANJA, "acesa — antes")):
    axa.add_collection3d(Poly3DCollection(fita(f_, y, suave(s_, 3), piso),
                                          alpha=.22, facecolor=cor, lw=0))
    axa.plot(f_, np.full_like(f_, y), suave(s_, 3), color=cor, lw=1.0,
             label=rot)
for k in idx:
    axa.plot([fr[k]] * 2, [2, 2], [v_antes[k] + 1.5, v_antes[k] + 6.0],
             color=LARANJA, lw=.8)
axa.text(fr[idx[len(idx) // 2]], 2, v_antes[ma].max() + 9,
         "%d máximos locais" % len(idx), color=LARANJA, fontsize=8.5,
         ha="center")
axa.add_collection3d(Poly3DCollection(chao(FCRIT[0], FCRIT[1], -.4, 2.4, piso),
                                      alpha=.16, facecolor=VERMELHO, lw=0))
moldura(axa, 2.4, (piso, -46.0), xticks=False)
axa.legend(loc="upper right", bbox_to_anchor=(.90, 1.015),
           bbox_transform=fig.transFigure, frameon=False, fontsize=8,
           handlelength=1.4, labelspacing=.2)

axb = fig.add_subplot(2, 1, 2, projection="3d")
pisob = -10.0
for y, f_, s_, cor in ((0, fd[cd], d_depois[cd], AZUL),
                       (1, fr[cr], d_antes[cr], LARANJA)):
    axb.add_collection3d(Poly3DCollection(fita(f_, y, suave(s_, 3), pisob),
                                          alpha=.22, facecolor=cor, lw=0))
    axb.plot(f_, np.full_like(f_, y), suave(s_, 3), color=cor, lw=1.0)
# a caixa do criterio, em toda a faixa varrida
for face in (chao(-6, 378, -.4, 1.4, CRIT), chao(-6, 378, -.4, 1.4, -CRIT)):
    axb.add_collection3d(Poly3DCollection(face, alpha=.17,
                                          facecolor=VERDE_LIMA,
                                          edgecolor=VERDE_LIMA, lw=.5))
axb.add_collection3d(Poly3DCollection(chao(FCRIT[0], FCRIT[1], -.4, 1.4, pisob),
                                      alpha=.16, facecolor=VERMELHO, lw=0))
axb.plot([], [], [], color=LARANJA, lw=1.2, label="antes da mitigação")
axb.plot([], [], [], color=AZUL, lw=1.2, label="depois")
axb.plot([], [], [], color=VERDE_LIMA, lw=3.0, alpha=.6,
         label="caixa do critério, %g dB" % CRIT)
moldura(axb, 1.4, (pisob, max(d_antes[ma].max(), CRIT) + 6))
axb.legend(loc="lower left", bbox_to_anchor=(.035, .012),
           bbox_transform=fig.transFigure, frameon=False, fontsize=8,
           handlelength=1.4, labelspacing=.2)
fig.text(.72, .045, "Frequência (MHz)", fontsize=9, color=TINTA, ha="center")
posiciona(axa, axb)
rotulos(fig, "(a)  Amplitude (dBm)", "(b)  Elevação Δ (dB)")
print("Figura 3 — versão isométrica, como publicada no artigo")
plt.show()

# ----------------------------------------------------------- versao plana
fig, (pa, pb) = plt.subplots(2, 1, figsize=(9.2, 5.6), sharex=True, dpi=110,
                             gridspec_kw=dict(hspace=.14))
for eixo in (pa, pb):
    eixo.axvspan(*FCRIT, color=VERMELHO, alpha=.07, lw=0)
    limpa(eixo)
pa.plot(fr[cr], suave(v_ref[cr], 3), color=CINZA, lw=.8,
        label="apagada — referência")
pa.plot(fd[cd], suave(v_dep[cd], 3), color=AZUL, lw=.9, label="acesa — depois")
pa.plot(fr[cr], suave(v_antes[cr], 3), color=VERMELHO, lw=.9,
        label="acesa — antes")
for k in idx:
    pa.plot([fr[k]] * 2, [v_antes[k] + 1.0, v_antes[k] + 4.0],
            color=VERMELHO, lw=.7)
pa.text(fr[idx[len(idx) // 2]], v_antes[ma].max() + 6,
        "%d máximos locais" % len(idx), color=VERMELHO, fontsize=8.5,
        ha="center")
pa.set_ylabel("Amplitude (dBm)")
pa.set_title("(a)", loc="left", fontsize=10, fontweight="bold")
pa.legend(loc="upper right", frameon=False, fontsize=8)

pb.axhline(0, color="#d5d8dd", lw=.7)
pb.plot(fd[cd], suave(d_depois[cd], 3), color=AZUL, lw=.9)
pb.plot(fr[cr], suave(d_antes[cr], 3), color=VERMELHO, lw=.9)
pb.axhline(CRIT, color="#4a4f57", lw=.8, ls=(0, (3.5, 2)))
pb.text(300, CRIT + 1.4, "critério %g dB" % CRIT, fontsize=8, color=TINTA,
        ha="center")
pb.set_ylabel("Δ (dB)")
pb.set_xlabel("Frequência (MHz)")
pb.set_title("(b)", loc="left", fontsize=10, fontweight="bold")
fig.tight_layout()
print("Figura 3 — versão plana, preservada no artigo como alternativa")
plt.show()

print("elevação média na faixa de análise:  antes %+.2f dB   depois %+.2f dB"
      % (m_a, m_d))


## Se alguma conferência reprovar

Abra uma *issue* com a linha divergente, a versão do `numpy` e o sistema
operacional. Discordância metodológica fundamentada é igualmente bem-vinda: o
critério de 3 dB e a escolha da faixa de análise são as duas decisões deste
trabalho que mais merecem escrutínio.